# Induced Matching Bootstrap

In this notebook, we consider the information given by subsamples to capture their topological quality using bootstrap of induced matchings.

We recycle code from https://gudhi.inria.fr/python/latest/rips_complex_sklearn_itf_ref.html

Here we compute landscapes of matching diagrams, the matched and unmatched part respectively, and compute their bootstrapped confidence intervals.

In [ ]:
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import os

plots_dir = os.path.join("plots", "matching-bootstrap")
os.makedirs(plots_dir, exist_ok=True)

In [ ]:
import tdqual.topological_data_quality_0 as tdqual

size_sample_X = 20

RandGen = np.random.default_rng(5)
# # Generate Random Sample
Z = tdqual.sampled_circle(0,2,40, RandGen)
Z = np.vstack((Z, tdqual.sampled_circle(0,2,70, RandGen) + [13,0]))
Z = np.vstack((Z, tdqual.sampled_circle(0,1,50, RandGen) + [5,10]))
Z = np.vstack((Z, tdqual.sampled_circle(0,1,2, RandGen) + [-5,10]))
Z = np.vstack((Z, tdqual.sampled_circle(0,1,2, RandGen) + [15,10]))
list_X_Z = []
# Take a bad subsample
list_X_Z.append((Z[:size_sample_X], Z))
#Take another bad subsample
# Take sample of points and reorder
X_indices = [0,40,110,160,162] + list(range(41,40+size_sample_X-5))
X_indices_compl = list(set(range(Z.shape[0]))-set(X_indices))
# Reorder Z and get X
X =Z[X_indices]
Z_new = np.vstack((X, Z[X_indices_compl]))
list_X_Z.append((X, Z_new))

num_samples = 3
for i in range(num_samples):
    # Take sample of points and reorder
    X_indices = RandGen.choice(Z.shape[0],size_sample_X, replace=False)
    X_indices_compl = list(set(range(Z.shape[0]))-set(X_indices))
    # Reorder Z and get X
    X =Z[X_indices]
    Z_new = np.vstack((X, Z[X_indices_compl]))
    list_X_Z.append((X, Z_new))

In [ ]:
# Plot point cloud
fig, ax = plt.subplots(ncols=1, figsize=(3,3))
X, Z = list_X_Z[1]
ax.scatter(X[:,0], X[:,1], color=mpl.colormaps["RdBu"](0.3/1.3), s=60, marker="o", zorder=2)
ax.scatter(Z[:,0], Z[:,1], color=mpl.colormaps["RdBu"](1/1.3), s=40, marker="x", zorder=1)
ax.set_axis_off()
plt.savefig(os.path.join(plots_dir, "points_0.png"))

Let us take boostrap samples for all points.

In [ ]:
from tdqual.matching_bootstrap import bootstrap_subsamples

# Constants for subsample
percentage_subsample = 0.7
X, Z = list_X_Z[0]
size_subsample_X = int(X.shape[0]*percentage_subsample)
size_subsample_compl = int((Z.shape[0]-X.shape[0])*percentage_subsample)
#Take subsamples
sub = []
for i in range(len(list_X_Z)):
    sub.append(bootstrap_subsamples(list_X_Z[i][0], list_X_Z[i][1], size_subsample_X, size_subsample_compl, nb_times=120))



In [ ]:
# Plot point cloud
fig, ax = plt.subplots(ncols=len(list_X_Z), figsize=(3*len(list_X_Z),3))

for i in range(len(sub)):
    X, Z = list_X_Z[i]
    ax[i].scatter(X[:,0], X[:,1], color=mpl.colormaps["RdBu"](0.3/1.3), s=60, marker="o", zorder=2)
    ax[i].scatter(Z[:,0], Z[:,1], color=mpl.colormaps["RdBu"](1/1.3), s=40, marker="x", zorder=1)
    #ax[i].set_axis_off()
    ax[i].set_title(f"Set {i}", color = mpl.colormaps["Set1"](i))

plt.savefig(os.path.join(plots_dir, "points-samples.png"))

In [ ]:
from scipy.stats import bootstrap

def get_CI(data, confidence_level=0.95):
    bootstrap_result = bootstrap((data,), np.mean, confidence_level=0.95, method='percentile')
    low = bootstrap_result.confidence_interval.low
    high = bootstrap_result.confidence_interval.high
    return low, high

In [ ]:
from tdqual.matching_bootstrap import process_pairs_L1

for i in range(len(sub)):
    finite_L1_weights, inf_L1_weights = process_pairs_L1(sub[i])
    low, high = get_CI(finite_L1_weights)
    print(f"Set {i}")
    print(f"finite:   [{low:5.3f} , {high:5.3f}] mean: {np.mean(finite_L1_weights):5.3f} ")
    low, high = get_CI(inf_L1_weights)
    print(f"infinite:   [{low:5.3f} , {high:5.3f}] mean: {np.mean(inf_L1_weights):5.3f} ")

Let us compute the confidence intervals for the first landscapes of all matching diagrams, separating the non-matched intervals with the matched intervals.

In [ ]:
from tdqual.matching_bootstrap import process_pairs_landscapes, plot_CI_landscape
import matplotlib.pyplot as plt
from gudhi.representations import Landscape

# We set num_landscapes=1 to match your original pipeline configuration
landscape_resolution = 600
landscape_transformer = Landscape(num_landscapes=1, resolution=landscape_resolution)
# Plot the CI of landscapes
fig, ax = plt.subplots(ncols=2, figsize=(10,5))
for i in range(len(sub)):
    finite_landscapes, inf_landscapes = process_pairs_landscapes(sub[i], landscape_transformer)
    plot_CI_landscape(finite_landscapes, mpl.colormaps["Set1"](i), f"Sub {i}", ax[0], landscape_resolution)
    plot_CI_landscape(inf_landscapes, mpl.colormaps["Set1"](i), f"Sub {i}", ax[1], landscape_resolution)

ax[0].set_title("Average landscapes (Finite part)")
ax[1].set_title("Average landscapes (Inf part)")
plt.legend()
plt.savefig(os.path.join(plots_dir, "CI-landscapes.png"))

# Bootstrap 2 subsamples

Try bootstrap of sample and total, and check how well they approximate their union. Given $X \subset Z$, we take a pair of bootstrap samples $X'$ and $Z'$ and compute both matchings $X' \hookrightarrow X'\cup Z'$ and also $Z' \hookrightarrow X'\cup Z'$ and average their $L^1$ norms along all samples.

In [ ]:
def bootstrap_subsamples_pairs(X, Z, nb_times=80):
    sub = []
    # construct a list of nb_times x nb_points
    for sub_idx in range(nb_times):
        X_idx = np.random.choice(range(X.shape[0]), X.shape[0])
        X_sub = X[X_idx]
        Z_idx = np.random.choice(range(Z.shape[0]), Z.shape[0])
        Z_sub = Z[Z_idx]
        # Append subsample 
        sub.append((X_sub, Z_sub))
    # range over all subsamples
    return sub

In [ ]:
size_subsamples = X.shape[0]
#Take subsamples
sub_pair = []
for i in range(len(list_X_Z)):
    sub_pair.append(bootstrap_subsamples_pairs(list_X_Z[i][0], list_X_Z[i][1], nb_times=200))

In [ ]:
means = []
for i in range(len(sub_pair)):
    means_sub = []
    print(f"Set {i}")
    print(f"Sub pair X")
    sub_pair_X = [(X, np.vstack((X,Z))) for (X,Z) in sub_pair[i]]
    finite_L1_weights, inf_L1_weights = process_pairs_L1(sub_pair_X)
    low, high = get_CI(finite_L1_weights)
    print(f"finite:   [{low:5.3f} , {high:5.3f}] mean: {np.mean(finite_L1_weights):5.3f} ")
    means_sub.append(np.mean(finite_L1_weights))
    low, high = get_CI(inf_L1_weights)
    print(f"infinite:   [{low:5.3f} , {high:5.3f}] mean: {np.mean(inf_L1_weights):5.3f} ")
    means_sub.append(np.mean(inf_L1_weights))
    print(f"Sub pair Z")
    sub_pair_Z = [(Z, np.vstack((Z,X))) for (X,Z) in sub_pair[i]]
    finite_L1_weights, inf_L1_weights = process_pairs_L1(sub_pair_Z)
    low, high = get_CI(finite_L1_weights)
    print(f"finite:   [{low:5.3f} , {high:5.3f}] mean: {np.mean(finite_L1_weights):5.3f} ")
    means_sub.append(np.mean(finite_L1_weights))
    low, high = get_CI(inf_L1_weights)
    print(f"infinite:   [{low:5.3f} , {high:5.3f}] mean: {np.mean(inf_L1_weights):5.3f} ")
    means_sub.append(np.mean(inf_L1_weights))
    means.append(means_sub)

In [ ]:
means = np.array(means)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


column_labels = ['fin_X_L1', 'inf_X_L1', 'fin_Z_L1', 'inf_Z_L1']
row_labels = [f'Sub_{i}' for i in range(means.shape[0])]

# Set figure size
plt.figure(figsize=(8, 6))

# Heatmap
sns.heatmap(means, 
            annot=True, 
            fmt=".2f", 
            xticklabels=column_labels, 
            yticklabels=row_labels, 
            cmap='coolwarm', 
            cbar=True)

# Labels
plt.title('Normas L1 de los subconjuntos', fontsize=14, pad=15)
plt.xticks(rotation=15) 
plt.yticks(rotation=0)

# Show heatmap
plt.tight_layout()
plt.show()